In [ ]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import sys
import os

load_dotenv(find_dotenv())

sys.path.append("..")
os.chdir("..")
os.getcwd()

In [ ]:
import requests
import base64

response = requests.post(
    os.getenv("AZURE_OPENAI_TTS_ENDPOINT"),
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.getenv('AZURE_OPENAI_API_KEY')}",
    },
    json={
        "input": "Hello, this is a text to speech test.",
        "voice": "alloy",
        "model": "tts",
    },
)

if response.status_code == 200:
    base64_string = base64.b64encode(response.content).decode("utf-8")

In [ ]:
f"data:audio/wav;base64,{base64_string}"

In [ ]:
from src.retrieval_graph import retrieval_graph

messages = [{"role": "user", "content": "tell me something about kangxi"}]
results = await retrieval_graph.ainvoke({"messages": messages})
results

## 验证 graph stream 并没有任何阻塞

In [ ]:
from src.graph import graph

events = []


def get_node_id(node, event):
    if event["name"] == node:
        return event["run_id"]
    else:
        return None


generator_id = None
async for event in graph.astream_events(
    {"messages": [{"role": "user", "content": "tell me something about porcelain"}]},
    version="v2",
):
    if generator_id is None:
        generator_id = get_node_id("generator", event)
    if event["event"] == "on_chat_model_stream" and generator_id in event["parent_ids"]:
        print(event["data"]["chunk"].content)

In [ ]:
def remove_unecessary_info(event):
    event.pop("metadata", None)
    return event


clean_events = list(map(remove_unecessary_info, events))
clean_events
# def get_node_events(node, events):
#     for i in range(len(events)):
#         if events[i]["name"] == node:
#             break

#     res = []
#     for i in range(i, len(events)):
#         if events[i]["event"] == "on_chain_end":
#             break
#         if events[i]["event"] == "on_chat_model_stream":
#             res.append(events[i])

#     return res

# def is_chat_chunk():


# wanted_events = get_node_events("generator", clean_events)
# wanted_events

In [ ]:
results["docs"][0].id

In [ ]:
from chromadb import PersistentClient

client = PersistentClient(path="./chroma_db")

In [ ]:
from pathlib import Path
import hashlib

data_folder = Path("./data/Objectifying_China/docs")


def get_file_id(file_path: str) -> str:
    return hashlib.md5(open(file_path, "rb").read()).hexdigest()


batch = {"ids": [], "documents": [], "metadatas": []}
for filepath in data_folder.glob("*.md"):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
        print(get_file_id(str(filepath)), filepath.name)
        batch["ids"].append(get_file_id(str(filepath)))
        batch["documents"].append(text)
        batch["metadatas"].append({"source": filepath.name})

In [ ]:
collection = client.get_or_create_collection(name="museum_knowledge_base")

In [ ]:
collection.add(**batch)